In [1]:
import re
import pandas as pd

In [2]:
def extract_amount(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return None
    
    lines = text.split('\n')
    for i, line in enumerate(lines):
        if re.search(r'(?:total amount due|total\s*\(\d+\)|amount due|amt due|grand total|\btotal\b)', line, re.IGNORECASE):
            for check_line in lines[i:i+2]:
                match = re.search(r'(\d{1,3}(?:,\d{3})*\.\d{2})', check_line)
                if match:
                    try:
                        val = float(match.group(1).replace(',', ''))
                        if val >= 5.0 and 1.0 <= val <= 99999.0:
                            return round(val, 2)
                    except:
                        continue
    
    blocklist = ['vat', 'vatable', 'subtotal', 'cash', 'change', 'discount', 'zero rated', 'exempt']
    candidates = []
    for match in re.finditer(r'\d{1,3}(?:,\d{3})*\.\d{2}', text):
        num_str = match.group(0)
        start = match.start()
        preceding = text[max(0, start-50):start].lower()
        if any(word in preceding for word in blocklist):
            continue
        try:
            val = float(num_str.replace(',', ''))
            if 1.0 <= val <= 99999.0:
                candidates.append(val)
        except:
            continue
    
    return round(max(candidates), 2) if candidates else None

In [3]:
def extract_date(text):
    if not isinstance(text, str):
        return None
    
    patterns = [
        r'(\d{2}/\d{2}/\d{4})',
        r'(\d{4}-\d{2}-\d{2})',
        r'(\d{1,2}-(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)-\d{4})'
    ]
    month_map = {
        'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
        'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12
    }
    
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            date_str = match.group(1)
            try:
                if '/' in date_str:
                    m, d, y = map(int, date_str.split('/'))
                elif len(date_str.split('-')) == 3 and date_str.split('-')[1].lower() in month_map:
                    parts = date_str.split('-')
                    d = int(parts[0])
                    m = month_map[parts[1].lower()]
                    y = int(parts[2])
                else:
                    y, m, d = map(int, date_str.split('-'))
                if 1 <= m <= 12 and 1 <= d <= 31 and 2020 <= y <= 2035:
                    return f"{y}-{m:02d}-{d:02d}"
            except:
                continue
    return None

In [4]:
def extract_store(text):
    if not isinstance(text, str):
        return None
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    for line in lines:
        if re.search(r'7-?Eleven|7-?Connect', line, re.IGNORECASE):
            clean = re.sub(r'Store#\d+|SN#.*|MIN#.*', '', line).strip()
            if len(clean) > 8:
                return clean
    
    for line in lines[:8]:
        if re.search(r'7-?Eleven|7-?Connect', line, re.IGNORECASE):
            return line.strip()
    
    for line in lines[:6]:
        if (5 <= len(line) <= 60 and
            re.search(r'[A-Za-z]{4,}', line) and
            not re.search(r'[@#|{}]', line) and
            not re.search(r'\d+\.\d{2}', line)):
            return line
    
    return None

In [5]:
df = pd.read_csv("../outputs/metrics/easyocr_results.csv")
ground_truth = pd.read_csv("../data/annotated/ground_truth.csv")

results = []

for _, row in df.iterrows():
    extracted_amount = extract_amount(row.get('extracted_text', ''))
    extracted_date = extract_date(row.get('extracted_text', ''))
    extracted_store = extract_store(row.get('extracted_text', ''))
    
    results.append({
        "filename": row["filename"],
        "condition": row["condition"],
        "extracted_amount": extracted_amount,
        "extracted_date": extracted_date,
        "extracted_store": extracted_store,
        "extracted_text": row.get("extracted_text", "")
    })

final_df = pd.DataFrame(results)
final_df = pd.merge(ground_truth, final_df, on=['filename', 'condition'])

In [6]:
final_df['amount_correct'] = abs(
    final_df['amount'] - final_df['extracted_amount'].fillna(-9999)
) < 2.0

final_df['date_correct'] = (
    final_df['date'].astype(str).str.strip() ==
    final_df['extracted_date'].astype(str).str.strip()
)

In [7]:
total = len(final_df)
amt_correct = final_df['amount_correct'].sum()
amt_extracted = final_df['extracted_amount'].notna().sum()
date_correct = final_df['date_correct'].sum()
date_extracted = final_df['extracted_date'].notna().sum()

print("=" * 90)
print("               IMPROVED TEXT-BASED POSTPROCESSING")
print("=" * 90)
print(final_df[[
    'filename', 'amount', 'extracted_amount', 'amount_correct',
    'date', 'extracted_date', 'date_correct'
]].to_string(index=False))

print()
print("─" * 50)
print(f"Amount Extracted  : {amt_extracted}/{total} ({amt_extracted/total*100:.1f}%)")
print(f"Amount Correct    : {amt_correct}/{total} ({amt_correct/total*100:.1f}%)")
print(f"Date Extracted    : {date_extracted}/{total} ({date_extracted/total*100:.1f}%)")
print(f"Date Correct      : {date_correct}/{total} ({date_correct/total*100:.1f}%)")
print("─" * 50)

final_df.to_csv("../outputs/metrics/final_extraction.csv", index=False)
print("\n✅ Results saved to outputs/metrics/final_extraction.csv")

               IMPROVED TEXT-BASED POSTPROCESSING
      filename  amount  extracted_amount  amount_correct       date extracted_date  date_correct
receipt_01.jpg    50.0              50.0            True 2026-05-10     2026-05-10          True
receipt_02.jpg   203.0             203.0            True 2026-05-10     2026-05-10          True
receipt_03.jpg    44.0              50.0           False 2026-05-10     2026-05-10          True
receipt_04.jpg    24.0              24.0            True 2026-05-11     2026-09-11         False
receipt_05.jpg  1300.0               NaN           False 2026-05-10            NaN         False
receipt_06.jpg  2000.0               NaN           False 2026-05-11     2020-08-01         False
receipt_07.jpg    11.0              11.0            True 2026-05-11     2026-05-11          True
receipt_08.jpg    73.0              73.0            True 2026-05-11     2026-08-11         False
receipt_09.jpg    10.0              10.0            True 2026-05-11          